<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 04 · REAL-TIME ANALYTICS WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">Model Data for Analytical Workloads</div>
  <p style="color:#475569;font-size:15px;line-height:1.7;max-width:900px;margin:0">Turn a source contract and analytical requirements into typed Doris tables with an explicit grain and Table Model.</p>
  <span style="display:inline-block;border:1px solid #99f6e4;border-radius:4px;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:14px;font-size:12px">Doris 4.1.3 · Grain · Data Types · NULL and DEFAULT · Duplicate Key · Aggregate Key</span>
</div>

By the end of this lab, you will have translated source and reporting requirements into two explicit grains, selected Doris data types and missing-value rules, and derived a detail model and a daily summary model from the questions each table must answer. Run the cells in order.

### Initialize the Lab

Run the next initialization cell before Section 1. It loads the shared course helper, installs the same output styles used in Level 1, and creates the `lab` object used by every later code cell.

Run it again whenever you restart the Jupyter kernel. It does **not** start Docker, create a table, or change data. Later cells can reconnect to an existing Doris sandbox. Do not continue until the green **Lab tools are ready** message appears.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT);


## 1. Inspect the source column contract

Module 4 builds new models from the Level 1 `events` table. First confirm that the complete source is available, then inspect the existing type, nullability, and default of each field.

The source contract states that identifiers are numeric, `event_time` supports time analysis, `revenue` requires exact decimal arithmetic, and `event_type` and `region` are bounded categories. Those requirements—not the largest value in this particular file—determine the durable column types. Section 4 verifies the row grain immediately before it selects the Table Model.

In [ ]:
lab.connect(container="doris", host="127.0.0.1", port=9030)
lab.execute("USE doris_course")

lab.sql("""
SELECT COUNT(*) AS source_rows
FROM events
""", title="Source availability");


**Expected result**

| source_rows |
|---:|
| 10,158,080 |

`source_rows` confirms that the complete Level 1 dataset is present. This check establishes source availability only; it does not yet determine what one row represents.

### Inspect the selected column contract

The metadata query exposes the existing Doris type, nullability, and default for each field. Read it as a contract: every choice should be justified by the source meaning or by an analytical operation, not merely copied from the current sample values.

In [ ]:
lab.sql(
    "SHOW FULL COLUMNS FROM events",
    title="Event-detail column contract",
    columns=["Field", "Type", "Null", "Default"],
);


**Expected result**

The metadata result should show the following contract:

| Field family | Selected contract | Requirement that justifies it |
|---|---|---|
| `event_id`, `user_id`, `product_id` | `BIGINT NOT NULL` | Numeric identifiers follow the upstream contract and retain growth headroom. |
| `event_time` | `DATETIME NOT NULL` | Queries filter, group, and order events by time. |
| `event_type` | `VARCHAR(32) NOT NULL` | A bounded category needs room for controlled future values. |
| `region` | `VARCHAR(16) NOT NULL` | A short, bounded category does not need an unbounded string. |
| `revenue` | `DECIMAL(12,2) NOT NULL` | Currency requires exact fixed-point arithmetic. |

The output verifies that the stored schema matches the selected contract. In particular, `DECIMAL(12,2)` supports exact currency arithmetic, while the bounded `VARCHAR` columns represent controlled categories. A type must cover the source contract and expected analytical operations; it should not be chosen by copying only the largest value observed today.

## 2. Observe the cost of a permissive source schema

A raw landing table may use `VARCHAR` to preserve source values before their meaning has been validated. That flexibility also allows numeric identifiers, timestamps, and revenue to enter Doris as text, so the schema cannot yet guarantee that they support the operations expected of them.

The sample contains five source rows. Four revenue values are valid decimal text; one is `not-a-number`. `region_text` and `product_id_text` are nullable, but the missing values have different business meanings: the event still needs a usable region, while a missing product is allowed.

The first result shows the work required when revenue remains text. `TRY_CAST` must be repeated in the analytical query, and failed conversions must be counted separately because `SUM` ignores the resulting `NULL`. The second result uses a deliberately large amount to make the difference between approximate `DOUBLE` arithmetic and exact `DECIMAL` arithmetic visible.

In [ ]:
lab.execute("""
CREATE TABLE IF NOT EXISTS modeling_raw_events (
    event_time_text VARCHAR(32) NOT NULL,
    event_id_text VARCHAR(32) NOT NULL,
    user_id_text VARCHAR(32) NOT NULL,
    event_type_text VARCHAR(32) NOT NULL,
    region_text VARCHAR(16) NULL,
    product_id_text VARCHAR(32) NULL,
    revenue_text VARCHAR(32) NOT NULL
)
DUPLICATE KEY(event_time_text, event_id_text)
DISTRIBUTED BY RANDOM BUCKETS 1
PROPERTIES ("replication_num" = "1")
""")

lab.execute("TRUNCATE TABLE modeling_raw_events")

lab.insert("""
INSERT INTO modeling_raw_events VALUES
    ('2020-03-01 09:00:00', '90001', '7001', 'view',     'region_01', '1005115', '0.00'),
    ('2020-03-01 09:01:00', '90002', '7001', 'cart',     'region_01', '1005115', '0.00'),
    ('2020-03-01 09:02:00', '90003', '7001', 'purchase', 'region_01', '1005115', '29.95'),
    ('2020-03-01 09:03:00', '90004', '7002', 'purchase', 'region_03', '13200021', 'not-a-number'),
    ('2020-03-01 09:04:00', '90005', '7003', 'purchase', NULL,        NULL,       '9.99')
""", title="Write the permissive source sample")

lab.sql("""
SELECT
    COUNT(*) AS raw_rows,
    SUM(TRY_CAST(revenue_text AS DECIMAL(12,2))) AS calculated_revenue,
    SUM(CASE
        WHEN TRY_CAST(revenue_text AS DECIMAL(12,2)) IS NULL THEN 1
        ELSE 0
    END) AS invalid_revenue_rows
FROM modeling_raw_events
""", title="Revenue calculated from text")

lab.sql("""
SELECT
    CAST('90071992547409.92' AS DOUBLE) AS float_amount,
    CAST('90071992547409.92' AS DOUBLE) + 0.01 AS float_plus_one_cent,
    CAST('90071992547409.92' AS DECIMAL(18,2)) + 0.01 AS decimal_plus_one_cent
""", title="Approximate and exact amount arithmetic");


**Expected result**

The raw table accepts all five rows, including `not-a-number`, because `VARCHAR` validates only that the value is text. It does not validate the business meaning of revenue.

| raw_rows | calculated_revenue | invalid_revenue_rows |
|---:|---:|---:|
| 5 | 39.94 | 1 |

`calculated_revenue` alone is incomplete evidence: `TRY_CAST` converts the malformed value to `NULL`, and `SUM` ignores it. The separate invalid-row count is therefore required wherever this text column is analyzed.

| float_amount | float_plus_one_cent | decimal_plus_one_cent |
|---:|---:|---:|
| 90071992547409.92 | 90071992547409.94 | 90071992547409.93 |

Adding one cent to the approximate `DOUBLE` value produces a two-cent change at this magnitude, while `DECIMAL(18,2)` produces the exact monetary result. The large value is used only to make binary floating-point precision visible; the event model continues to use the smaller `DECIMAL(12,2)` contract required by its revenue range.

The raw schema also makes `region_text` nullable even though every analytical event needs a usable region. Leaving that rule unresolved would force later queries to decide repeatedly whether a missing region should be rejected, grouped as `NULL`, or replaced.

## 3. Establish the typed contract at ingestion

The internal table assigns one business meaning to each column: identifiers use `BIGINT`, time uses `DATETIME`, and revenue uses exact `DECIMAL(12,2)`. Short categorical fields use bounded `VARCHAR` lengths rather than a generic oversized string. Because one row still represents one original event, the table uses the Duplicate Key model.

Conversion and validation now happen once while rows cross from the permissive source into the typed table. `TRY_CAST(revenue_text AS DECIMAL(12,2)) IS NOT NULL` excludes the malformed revenue row; subsequent analytical queries do not repeat that conversion.

Two inserts are required because `DEFAULT` and `NULL` mean different things:

1. Rows that already have a region explicitly write `region_text` into the `region` column.
2. The valid row with no region omits `region` from its target column list. Doris therefore supplies `DEFAULT "unknown"`. Explicitly writing `NULL` would violate `region NOT NULL`; a default is applied when the column is omitted, not whenever a supplied value is `NULL`.

`product_id` remains nullable because an event without a product is valid under this contract. The schema therefore removes unnecessary nullability from `region` without turning every missing value into an error.

In [ ]:
lab.execute("""
CREATE TABLE IF NOT EXISTS modeling_typed_events (
    event_time DATETIME NOT NULL,
    event_id BIGINT NOT NULL,
    user_id BIGINT NOT NULL,
    event_type VARCHAR(32) NOT NULL,
    region VARCHAR(16) NOT NULL DEFAULT "unknown",
    product_id BIGINT NULL,
    revenue DECIMAL(12,2) NOT NULL DEFAULT "0.00"
)
DUPLICATE KEY(event_time, event_id)
DISTRIBUTED BY RANDOM BUCKETS 1
PROPERTIES ("replication_num" = "1")
""")

lab.execute("TRUNCATE TABLE modeling_typed_events")

lab.insert("""
INSERT INTO modeling_typed_events (
    event_time, event_id, user_id, event_type, region, product_id, revenue
)
SELECT
    CAST(event_time_text AS DATETIME),
    CAST(event_id_text AS BIGINT),
    CAST(user_id_text AS BIGINT),
    event_type_text,
    region_text,
    CAST(product_id_text AS BIGINT),
    TRY_CAST(revenue_text AS DECIMAL(12,2))
FROM modeling_raw_events
WHERE region_text IS NOT NULL
  AND TRY_CAST(revenue_text AS DECIMAL(12,2)) IS NOT NULL
""", title="Write valid rows with an explicit region")

lab.insert("""
INSERT INTO modeling_typed_events (
    event_time, event_id, user_id, event_type, product_id, revenue
)
SELECT
    CAST(event_time_text AS DATETIME),
    CAST(event_id_text AS BIGINT),
    CAST(user_id_text AS BIGINT),
    event_type_text,
    CAST(product_id_text AS BIGINT),
    TRY_CAST(revenue_text AS DECIMAL(12,2))
FROM modeling_raw_events
WHERE region_text IS NULL
  AND TRY_CAST(revenue_text AS DECIMAL(12,2)) IS NOT NULL
""", title="Omit region to apply its default")

lab.sql(
    "SHOW FULL COLUMNS FROM modeling_typed_events",
    title="Typed column contract",
    columns=["Field", "Type", "Null", "Key", "Default"],
)

lab.sql("""
SELECT
    COUNT(*) AS typed_rows,
    SUM(revenue) AS typed_revenue,
    SUM(CASE WHEN region = 'unknown' THEN 1 ELSE 0 END) AS defaulted_regions,
    SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS null_product_ids
FROM modeling_typed_events
""", title="Typed-table calculation");


**Expected result**

The metadata output should make the boundary visible:

| Source representation | Typed internal-table contract | Effect after ingestion |
|---|---|---|
| IDs stored in generic text columns | `BIGINT NOT NULL` | Numeric identity no longer requires a runtime cast |
| Timestamp stored as text | `DATETIME NOT NULL` | Time filtering and ordering use a temporal type |
| Revenue stored as text | `DECIMAL(12,2) NOT NULL` | Monetary aggregation uses exact fixed-point values |
| `region_text` allows `NULL` | `region NOT NULL DEFAULT "unknown"` | Every stored event has a usable region category |
| `product_id_text` allows `NULL` | `product_id BIGINT NULL` | Missing product identity remains valid and distinguishable |

The typed-table calculation should report:

| typed_rows | typed_revenue | defaulted_regions | null_product_ids |
|---:|---:|---:|---:|
| 4 | 39.94 | 1 | 1 |

The counts reconcile the transition: five raw rows minus one malformed revenue row equals four typed rows. One of those rows receives the region default, while its optional product remains `NULL`.

The analytical expression is now simply `SUM(revenue)`. Step 2 needed `SUM(TRY_CAST(revenue_text AS DECIMAL(12,2)))` plus an invalid-row count because it postponed validation until query time. The typed table performs that work once during ingestion and gives later queries a stable contract.

## 4. Build the complete query-ready event model

The five-row example established the column contract. The source contract defines one row as one original event, so the final model preserves event-detail grain and applies the typed contract to all 10,158,080 rows. Grain is the business meaning of a row; it is not inferred by building an expensive distinct set over the entire table. The reconciliation after insertion compares row count and revenue to verify that the complete event-detail dataset was copied.

The model also adds physical design choices derived from the event workload:

| Workload requirement | Table decision |
|---|---|
| Preserve every original event | Duplicate Key model |
| Filter event history by time and retain data by calendar period | Monthly Auto Range Partition on `event_time` |
| Make time-range scans benefit from ordered storage | Sort key begins with `event_time` |
| Distribute a user's events predictably and support parallel scans | `HASH(user_id)` with 10 Buckets per Partition |
| Run in the single-BE course sandbox | One replica |

`events_modelled` remains at event-detail grain: one row still represents one original event. Because repeated Key values represent event history rather than replacement or aggregation, the table uses the Duplicate Key model. Partition, Bucket, and sort key change the physical organization; they do not change the row meaning or remove data. The existing target is truncated before the complete insert, so rerunning the cell produces the same table contents.

In [ ]:
lab.execute("""
CREATE TABLE IF NOT EXISTS events_modelled (
    event_time DATETIME NOT NULL,
    event_id BIGINT NOT NULL,
    user_id BIGINT NOT NULL,
    event_type VARCHAR(32) NOT NULL,
    region VARCHAR(16) NOT NULL,
    product_id BIGINT NOT NULL,
    revenue DECIMAL(12,2) NOT NULL DEFAULT "0.00"
)
DUPLICATE KEY(event_time, event_id, user_id)
AUTO PARTITION BY RANGE (date_trunc(event_time, 'month'))
()
DISTRIBUTED BY HASH(user_id) BUCKETS 10
PROPERTIES ("replication_num" = "1")
""")

lab.execute("TRUNCATE TABLE events_modelled")

lab.insert("""
INSERT INTO events_modelled (
    event_time, event_id, user_id, event_type, region, product_id, revenue
)
SELECT
    event_time, event_id, user_id, event_type, region, product_id, revenue
FROM events
""", title="Build the complete query-ready event model")

lab.sql("""
SELECT
    'events' AS table_name,
    COUNT(*) AS row_count,
    SUM(revenue) AS total_revenue
FROM events
UNION ALL
SELECT
    'events_modelled',
    COUNT(*),
    SUM(revenue)
FROM events_modelled
ORDER BY table_name
""", title="Source and final-model reconciliation");

**Expected result**

The source and final model should reconcile in one result:

| table_name | row_count | total_revenue |
|---|---:|---:|
| events | 10,158,080 | 39,984,455.64 |
| events_modelled | 10,158,080 | 39,984,455.64 |

The source contract establishes that one row represents one original event. Matching row count and revenue across the two tables verifies that the complete insert did not aggregate or omit the event data. `events_modelled` remains available as the query-ready event-detail table for Modules 5 and 6.

## 5. Translate a reporting requirement into a summary model

The reporting requirement is: **return daily event count and revenue by region and event type without scanning individual events for every report**. Start with that question, then derive the table instead of starting with Key syntax.

| Reporting requirement | Modeling decision |
|---|---|
| Group reports by date, region, and event type | One row represents one `(event_date, region, event_type)` combination. |
| Count events and total revenue | Store additive `event_count` and `total_revenue` measures with `SUM`. |
| Individual event and user identifiers are not queried from this summary | Do not carry `event_id`, `user_id`, or `product_id` into this grain. |
| Later batches can contain an existing date-region-type combination | Use the Aggregate Key model so repeated combinations merge through their declared aggregation functions. |

Grain and Key columns are related, but they are not the same concept. Grain is the business meaning of one row. Key columns implement that meaning in the table definition. Here, `event_date`, `region`, and `event_type` become the Aggregate Key because together they identify one row at the chosen reporting grain.

This differs from the event-detail model created earlier. That model uses Duplicate Key because each valid original event must remain available. A current-state model would instead use Unique Key when one business key must expose only its latest row. The choice follows what a row represents and what repeated keys mean; it does not follow from the number of Key columns.

| Repeated-key requirement | Table Model | Responsibility of Key columns |
|---|---|---|
| Preserve every original row when the same Key values appear again | Duplicate Key | Define the sort key; repeated-key rows remain visible |
| Expose one current row for each business key | Unique Key | Define the sort key and logical uniqueness used for upsert |
| Combine measures when the same grouping dimensions appear again | Aggregate Key | Define the sort key and aggregation group; Value columns merge with their declared aggregation functions |

Do not define `user_count` as an ordinary `SUM` value: distinct users are not additive because one user can appear in multiple input batches. Use a Bitmap or HLL state when a model requires a mergeable distinct count.

The Aggregate Key columns also form the sort key. The table uses `HASH(region)` distribution with one Bucket because this course sandbox has one BE and the resulting rollup is small. It has no explicit Partition because this small summary has no independent Partition-level lifecycle requirement. These choices demonstrate that Partition, Bucket, and sort key should follow data volume, filtering, distribution, and lifecycle requirements rather than a universal template.

In [ ]:
lab.execute("""
CREATE TABLE IF NOT EXISTS daily_event_metrics (
    event_date DATE NOT NULL,
    region VARCHAR(16) NOT NULL,
    event_type VARCHAR(32) NOT NULL,
    event_count BIGINT SUM NOT NULL DEFAULT "0",
    total_revenue DECIMAL(18,2) SUM NOT NULL DEFAULT "0.00"
)
AGGREGATE KEY(event_date, region, event_type)
DISTRIBUTED BY HASH(region) BUCKETS 1
PROPERTIES ("replication_num" = "1")
""")

lab.execute("TRUNCATE TABLE daily_event_metrics")

lab.insert("""
INSERT INTO daily_event_metrics (
    event_date, region, event_type, event_count, total_revenue
)
SELECT
    TO_DATE(event_time),
    region,
    event_type,
    COUNT(*),
    SUM(revenue)
FROM events_modelled
GROUP BY TO_DATE(event_time), region, event_type
""", title="Build the daily reporting grain")

lab.sql("""
SELECT
    'events_modelled' AS table_name,
    COUNT(*) AS stored_rows,
    COUNT(*) AS represented_event_rows,
    SUM(revenue) AS represented_revenue
FROM events_modelled
UNION ALL
SELECT
    'daily_event_metrics',
    COUNT(*),
    SUM(event_count),
    SUM(total_revenue)
FROM daily_event_metrics
ORDER BY table_name
""", title="Detail and summary model reconciliation");


**Expected result**

The result mirrors the reconciliation in Section 4 while making the change in grain visible:

| table_name | stored_rows | represented_event_rows | represented_revenue |
|---|---:|---:|---:|
| daily_event_metrics | 280 | 10,158,080 | 39,984,455.64 |
| events_modelled | 10,158,080 | 10,158,080 | 39,984,455.64 |

`events_modelled` stores one row per original event, so its stored and represented row counts are the same. `daily_event_metrics` stores only 280 date-region-event type combinations, but its additive measures still represent all 10,158,080 events and the same total revenue. The smaller `stored_rows` value verifies the summary grain; matching represented totals verifies that the aggregation preserved the declared measures.

The summary supports questions that can be answered from `event_date`, `region`, `event_type`, `event_count`, and `total_revenue`. It cannot recover an individual event, user, product, or distinct-user count because those fields are outside its grain. Keeping event detail and daily metrics in separate tables avoids mixing rows that represent different business objects.

### Stop the Doris sandbox

Run this optional cell to release CPU and memory. It stops Doris but preserves the container, image, named volumes, and Lab 4 tables.

In [ ]:
lab.shell(r"""
set -euo pipefail

docker stop doris
docker inspect --format 'container={{.State.Status}}' doris
""", title="Stop the Doris sandbox");


**Expected result:** Docker reports `container=exited`. The Level 1 baseline and Lab 4 tables remain in the named volumes.

### Restart the Doris sandbox

Run this cell to continue with the same environment. It starts the existing container, reconnects to FE, and verifies the persisted Lab 4 results.

In [ ]:
lab.shell(r"""
set -euo pipefail

docker start doris
docker inspect --format 'container={{.State.Status}} health={{.State.Health.Status}}' doris
""", title="Start the Doris sandbox")

lab.connect(container="doris", host="127.0.0.1", port=9030)
lab.execute("USE doris_course")

lab.sql("""
SELECT
    (SELECT COUNT(*) FROM events_modelled) AS modelled_event_rows,
    (SELECT SUM(event_count) FROM daily_event_metrics) AS represented_event_rows,
    (SELECT SUM(total_revenue) FROM daily_event_metrics) AS represented_revenue
""", title="Recovered Lab 4 models");


**Expected result:** the container returns to `healthy`. The result reports 10,158,080 modelled event rows, 10,158,080 represented event rows, and represented revenue of 39,984,455.64.

## Lab complete

You translated a source contract into a complete query-ready event model, then translated a reporting requirement into a daily summary grain. The Table Models, Key columns, additive measures, nullability, defaults, distribution, Bucket count, and Partition choices followed from those requirements. Finally, you reconciled the summary with its detail source for a grain-compatible business question.

Official references: [Table Model Overview](https://doris.apache.org/docs/4.x/table-design/data-model/intro/) · [Table Model Best Practices](https://doris.apache.org/docs/4.x/table-design/data-model/tips/) · [CREATE TABLE](https://doris.apache.org/docs/4.x/sql-manual/sql-statements/table-and-view/table/CREATE-TABLE/) · [CAST and TRY_CAST](https://doris.apache.org/docs/4.x/sql-manual/basic-element/sql-data-types/conversion/cast-expr/) · [Aggregate Key model](https://doris.apache.org/docs/4.x/table-design/data-model/aggregate/)